# 05 · Silhouette and agreement with CDR

These are reported for completeness and are not used to select configurations.

1. Silhouette width, computed in each configuration's own distance. Because it depends on the distance, it can
   only be compared within the same distance family, so the values are grouped by family.
2. ARI and NMI against CDR. CDR has four levels (0, 0.5, 1, 2), but only 2 of the 216 patients have CDR = 2, and
   a partition scored against more categories than it can represent is penalised for that alone. CDR is
   therefore merged in two ways:
   - 3 categories: 0 / 0.5 / ≥ 1,
   - 2 categories: healthy (CDR 0) vs. impaired (CDR ≥ 0.5).

   C2 and M3 use CDR to build the clusters, so their agreement with CDR is partly built in.

In [1]:
import json

%run -i ../src/setup.ipynb

from sklearn.metrics import silhouette_score, adjusted_rand_score, normalized_mutual_info_score

CMP = cfg.RESULTS_DIR / "02_compare_configurations"
OUT = cfg.RESULTS_DIR / "05_secondary_results"
OUT.mkdir(parents=True, exist_ok=True)

selection = json.loads((CMP / "selection.json").read_text())
K = selection["k"]
labels = pd.read_csv(CMP / "labels.csv")
df = load_analysis_data()
assert (labels["patient_id"] == df["patient_id"]).all()
ORDER = ["C1", "C1-b", "C2", "I1", "I2", "M1", "M2", "M3"]

## 1. Silhouette by distance family

In [2]:
FAMILY = {"ggower": "Robust G-Gower", "relms": "RelMS", None: "Euclidean (k-means)"}
rows = []
for name in ORDER:
    metric = selection["m3_metric"] if name == "M3" else cfg.CONFIGS[name].get("metric")
    fit = fit_configuration(name, K, df, metric=metric if name == "M3" else None)
    lab = labels[name].to_numpy()
    if fit["D"] is None:
        s = silhouette_score(fit["X"], lab, metric="euclidean")
    else:
        s = silhouette_score(fit["D"], lab, metric="precomputed")
    rows.append({"distance family": FAMILY[metric], "configuration": name,
                 "feature space": cfg.CONFIGS[name]["feature_space"], "silhouette": round(s, 3)})
sil = pd.DataFrame(rows).sort_values(["distance family", "configuration"]).set_index(["distance family", "configuration"])
sil.to_csv(OUT / "silhouette_by_family.csv")
sil

feature space  silhouette
distance family     configuration                                     
Euclidean (k-means) C1-b                          Clinical       0.257
RelMS               I1                              Images       0.020
                    M1                   Images + clinical       0.022
Robust G-Gower      C1                            Clinical       0.140
                    C2                      Clinical + CDR       0.112
                    I2                              Images       0.023
                    M2                   Images + clinical       0.021
                    M3             Images + clinical + CDR       0.020

## 2. Agreement with CDR

In [3]:
cdr3 = np.select([df["CDR"] == 0, df["CDR"] == 0.5], ["0", "0.5"], ">=1")
cdr2 = np.where(df["CDR"] == 0, "healthy", "impaired")
print("3 categories:", pd.Series(cdr3).value_counts().to_dict())
print("2 categories:", pd.Series(cdr2).value_counts().to_dict())

rows = []
for name in ORDER:
    lab = labels[name].to_numpy()
    rows.append({"configuration": name,
                 "CDR is an input": "yes" if name in ("C2", "M3") else "no",
                 "ARI (0 / 0.5 / ≥1)": adjusted_rand_score(cdr3, lab),
                 "NMI (0 / 0.5 / ≥1)": normalized_mutual_info_score(cdr3, lab),
                 "ARI (healthy / impaired)": adjusted_rand_score(cdr2, lab),
                 "NMI (healthy / impaired)": normalized_mutual_info_score(cdr2, lab)})
agree = pd.DataFrame(rows).set_index("configuration").round(3)
agree.to_csv(OUT / "agreement_with_cdr.csv")
agree

3 categories: {'0': 133, '0.5': 57, '>=1': 26}
2 categories: {'healthy': 133, 'impaired': 83}


,CDR is an input,ARI (0 / 0.5 / ≥1),NMI (0 / 0.5 / ≥1),ARI (healthy / impaired),NMI (healthy / impaired)
configuration,,,,,
C1,no,0.093,0.184,0.126,0.189
C1-b,no,0.133,0.249,0.124,0.226
C2,yes,0.177,0.340,0.205,0.316
I1,no,-0.002,0.018,0.003,0.015
I2,no,0.010,0.030,0.019,0.030
M1,no,0.030,0.067,0.052,0.068
M2,no,0.030,0.038,0.034,0.033
M3,yes,0.053,0.070,0.061,0.060


## 3. Interpretation

Silhouette: within the Robust G-Gower family, the clinical configurations (C1 0.14, C2 0.11) have clearly higher
silhouettes than those with images (about 0.02); the image blocks seem to add spread rather than group structure.
The RelMS configurations are also about 0.02. C1-b's 0.26 is computed in a different (Euclidean) space and cannot be
compared with the others. All values are low: the patients form a continuum rather than separate groups.

Agreement with CDR is modest everywhere (ARI ≤ 0.21, NMI ≤ 0.34). This is expected: the clusters are profiles
built from several dimensions (age, brain volume, education, cognition), severity being only one of them, and
four clusters cannot reproduce a three-level scale even when one of them is a clear impaired group (C1-b's cluster
4: 97% with CDR > 0). The clinical configurations agree most, C2 in particular since it uses CDR; the image-only
configurations not at all (ARI ≈ 0).